# 📓 03. Normalization

### 🎯 Objective: Apply uniform MinMaxScaler to all telemetry features.



### 🧹 Output Artifacts:
- `../../data/processed/3_normalized_telemetry.csv`


In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

INPUT_PATH = '../../data/processed/2_cleaned_telemetry_for_modelling.csv'
OUTPUT_PATH = '../../data/processed/3_normalized_telemetry.csv'

print("Libraries imported")

Libraries imported


In [2]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


In [3]:
# ── v2.2: Derived Features ──────────────────────────────────────────────────
# damagePerHit: weapon-class-agnostic combat intensity signal.
# Sniper/heavy-weapon players hit few enemies but deal high damage per hit.
# Without this, raw enemiesHit alone underrepresents their combat engagement.
df['damage_per_hit'] = df['damageDone'] / df['enemiesHit'].clip(lower=1)

# pickupAttemptRate: deliberateness of collection intent.
# True collectors actively attempt pickups (high rate);
# explorers pass near items incidentally (low rate).
df['pickup_attempt_rate'] = df['pickupAttempts'] / df['timeNearInteractables'].clip(lower=1)

print('Derived features added:')
print(f'  damage_per_hit range: [{df["damage_per_hit"].min():.3f}, {df["damage_per_hit"].max():.3f}]')
print(f'  pickup_attempt_rate range: [{df["pickup_attempt_rate"].min():.3f}, {df["pickup_attempt_rate"].max():.3f}]')


Derived features added:
  damage_per_hit range: [0.000, 18.667]
  pickup_attempt_rate range: [0.000, 42.000]


In [4]:
# Define features to normalize
features_to_normalize = [
    'enemiesHit', 'damageDone', 'timeInCombat', 'kills',
    'itemsCollected', 'pickupAttempts', 'timeNearInteractables',
    'distanceTraveled', 'timeSprinting', 'timeOutOfCombat',
    'damage_per_hit',       # v2.2 derived
    'pickup_attempt_rate',  # v2.2 derived
]

available_features = [f for f in features_to_normalize if f in df.columns]
print(f"Features to normalize: {len(available_features)}")

Features to normalize: 12


## 🔥 CRITICAL BUG HISTORY: The 'No Normalization' Incident

> **Date:** January 27, 2026
>
> **The Bug:** In Phase 1, we implemented `df.fillna(0)` but forgot to actually call the scaler. The code looked correct visually but wasn't transforming values.
>
> **The Impact:**
> * Features like `distanceTraveled` (Range: 0-5000) completely dominated Euclidean distance calculations.
> * Features like `kills` (Range: 0-10) were mathematically invisible.
> * **Result:** The algorithm categorized 98.6% of gameplay as "Exploration" purely due to the distance magnitude.
>
> **The Fix:** We now strictly apply `MinMaxScaler` to force all 10 features into the `[0, 1]` range, ensuring equal weighting.



## 📜 Optimization Study: Why MinMaxScaler?

> **Grid Search Results:** We tested three normalization strategies to find the optimal input format:
>
> 1. **MinMaxScaler (Selected):** Best stability. Preserves the exact `[0, 1]` bounds required for neural networks.
> 2. **Log-Sparse Scaler:** Tested to handle the "power law" distribution of combat events. Showed negligible improvement (+0.4% Silhouette) but destroyed interpretability.
> 3. **RobustScaler:** Rejected because it doesn't provide fixed bounds, risking gradient explosion in the ANFIS surrogate.



In [5]:
# Apply MinMaxScaler
scaler = MinMaxScaler()
df[available_features] = scaler.fit_transform(df[available_features].fillna(0))

print("Applied uniform MinMaxScaler [0, 1]")
print(f"Normalized features:\n{df[available_features].describe()}")

Applied uniform MinMaxScaler [0, 1]
Normalized features:
        enemiesHit   damageDone  timeInCombat        kills  itemsCollected  \
count  3240.000000  3240.000000   3240.000000  3240.000000     3240.000000   
mean      0.033951     0.039193      0.179007     0.029918        0.094255   
std       0.061398     0.064511      0.223110     0.058657        0.131374   
min       0.000000     0.000000      0.000000     0.000000        0.000000   
25%       0.000000     0.000000      0.000000     0.000000        0.000000   
50%       0.000000     0.000000      0.065574     0.000000        0.076923   
75%       0.053191     0.074627      0.327869     0.066667        0.153846   
max       1.000000     1.000000      1.000000     1.000000        1.000000   

       pickupAttempts  timeNearInteractables  distanceTraveled  timeSprinting  \
count     3240.000000            3240.000000       3240.000000    3240.000000   
mean         0.044323               0.084534          0.455634       0.174519 

In [6]:
# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Saved to ../../data/processed/3_normalized_telemetry.csv


In [7]:
import json
import os

# Export Scaler Parameters for Frontend Integration (v2.2)
MODEL_DIR = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

scaler_params = {
    'features': list(available_features),
    'data_min': scaler.data_min_.tolist(),
    'data_max': scaler.data_max_.tolist(),
    'data_range': (scaler.data_max_ - scaler.data_min_).tolist(),
    'min_value': float(scaler.feature_range[0]),
    'max_value': float(scaler.feature_range[1])
}

SCALER_OUT = os.path.join(MODEL_DIR, 'scaler_params.json')
with open(SCALER_OUT, 'w') as f:
    json.dump(scaler_params, f, indent=2)

print(f'Scaler parameters exported to {SCALER_OUT}')
print(f'Features included ({len(available_features)}): {available_features}')


Scaler parameters exported to ../../data/models\scaler_params.json
Features included (12): ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'distanceTraveled', 'timeSprinting', 'timeOutOfCombat', 'damage_per_hit', 'pickup_attempt_rate']
